In [0]:

import requests
import hashlib
import shutil
import csv
import re

from pathlib import Path
from zipfile import ZipFile, BadZipFile
from pyspark.sql import functions as F

BASE_URL = (
    "https://repositoriodeis.minsal.cl/"
    "DatosAbiertos/EGRESOS"
)

YEARS = [2018, 2019]

BASE = Path(
    "/Volumes/workspace/codi_bronze/raw_files"
)

ZIP_DIR = BASE / "source_zip"
EXTRACT_DIR = BASE / "extracted"

ZIP_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

def hash_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)
    return h.hexdigest()

def download_zip(year):
    url = f"{BASE_URL}/EGRESOS_{year}.zip"
    path = ZIP_DIR / f"EGRESOS_{year}.zip"

    if path.exists():
        if ZipFile(path).testzip() is None:
            print(f"{year}: ZIP existente y válido")
            return path

    temp = path.with_suffix(".zip.part")

    try:
        with requests.get(
            url, stream=True, timeout=(20, 120)
        ) as response:

            response.raise_for_status()

            with open(temp, "wb") as output:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        output.write(chunk)

        with ZipFile(temp) as zf:
            if zf.testzip() is not None:
                raise ValueError("ZIP corrupto")

        temp.replace(path)

    finally:
        if temp.exists():
            temp.unlink()

    print(f"{year}: descarga OK")
    return path

def extract_zip(year, path):
    folder = EXTRACT_DIR / str(year)
    folder.mkdir(parents=True, exist_ok=True)

    csv_files = []
    seen_names = set()

    with ZipFile(path) as zf:
        for item in zf.infolist():
            if item.is_dir():
                continue

            name = Path(item.filename).name

            if not name.lower().endswith(
                (".csv", ".xlsx", ".xls", ".txt")
            ):
                continue

            if name in seen_names:
                raise ValueError(
                    f"Nombre duplicado en ZIP: {name}"
                )
            seen_names.add(name)

            target = folder / name

            # Comparar contenido para evitar
            # reutilizar archivos incompletos
            if target.exists():
                if (
                    target.stat().st_size == item.file_size
                    and (hash_file(target) ==
                         hashlib.sha256(
                             zf.read(item)
                         ).hexdigest())
                ):
                    pass
                else:
                    raise ValueError(
                        f"Archivo distinto ya existente: {target}"
                    )
            else:
                with zf.open(item) as source:
                    with open(target, "wb") as dest:
                        shutil.copyfileobj(source, dest)

            if name.lower().endswith(".csv"):
                csv_files.append(target)

    return csv_files

def detect_csv(path):
    with open(path, "rb") as f:
        sample_bytes = f.read(65536)

    for encoding in ["utf-8-sig", "cp1252"]:
        try:
            sample = sample_bytes.decode(encoding)
            break
        except UnicodeDecodeError:
            continue
    else:
        encoding = "latin-1"
        sample = sample_bytes.decode(encoding)

    dialect = csv.Sniffer().sniff(
        sample, delimiters=";,\t|"
    )

    spark_encoding = {
        "utf-8-sig": "UTF-8",
        "cp1252": "windows-1252",
        "latin-1": "ISO-8859-1"
    }[encoding]

    return dialect.delimiter, spark_encoding


In [0]:

import uuid

resultados = []

for year in YEARS:

    print(f"\n{'=' * 55}")
    print(f"PROCESANDO DEIS {year}")
    print("=" * 55)

    table = f"workspace.codi_bronze.deis_egresos_{year}"

    # No modificar tablas existentes
    if spark.catalog.tableExists(table):
        print(f"Tabla existente: {table}")
        resultados.append({
            "year": year,
            "status": "YA_EXISTE",
            "records": spark.table(table).count()
        })
        continue

    try:
        zip_path = download_zip(year)
        csv_files = extract_zip(year, zip_path)

        if len(csv_files) != 1:
            raise ValueError(
                f"Se esperaban 1 CSV, "
                f"se encontraron {len(csv_files)}"
            )

        csv_path = csv_files[0]

        delimiter, encoding = detect_csv(csv_path)

        df = (
            spark.read
            .format("csv")
            .option("header", "true")
            .option("sep", delimiter)
            .option("encoding", encoding)
            .option("inferSchema", "false")
            .option("mode", "FAILFAST")
            .load(str(csv_path))
        )

        if len(df.columns) <= 1:
            raise ValueError("Estructura CSV inválida")

        source_count = df.count()

        if source_count == 0:
            raise ValueError("CSV sin registros")

        batch_id = str(uuid.uuid4())

        df_bronze = (
            df
            .withColumn("_source_system", F.lit("DEIS"))
            .withColumn("_source_year", F.lit(year))
            .withColumn(
                "_source_file", F.lit(str(csv_path))
            )
            .withColumn(
                "_source_sha256",
                F.lit(hash_file(csv_path))
            )
            .withColumn("_batch_id", F.lit(batch_id))
            .withColumn(
                "_ingestion_timestamp",
                F.current_timestamp()
            )
        )

        (
            df_bronze.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(table)
        )

        target_count = spark.table(table).count()

        if target_count != source_count:
            raise RuntimeError(
                f"Descuadre: {source_count} "
                f"vs {target_count}"
            )

        resultados.append({
            "year": year,
            "status": "OK",
            "records": target_count
        })

        print(f"{year}: {target_count:,} registros")

    except Exception as e:
        resultados.append({
            "year": year,
            "status": "ERROR",
            "records": 0
        })
        print(f"ERROR {year}: {type(e).__name__}: {e}")

display(spark.createDataFrame(resultados))


In [0]:

YEARS_COMPARE = [2018, 2019, 2020]

tables = {}
summary = []

metadata = {
    "_source_system",
    "_source_year",
    "_source_file",
    "_source_sha256",
    "_batch_id",
    "_ingestion_timestamp"
}

for year in YEARS_COMPARE:

    table = f"workspace.codi_bronze.deis_egresos_{year}"

    if not spark.catalog.tableExists(table):
        print(f"Tabla no encontrada: {table}")
        continue

    df = spark.table(table)

    columns = [
        c for c in df.columns
        if c not in metadata
    ]

    tables[year] = set(columns)

    summary.append({
        "year": year,
        "records": df.count(),
        "source_columns": len(columns)
    })

display(spark.createDataFrame(summary))

# Comparación de esquema por columna
if tables:
    all_columns = sorted(
        set.union(*tables.values())
    )

    comparison = [
        {
            "column": col,
            **{
                f"year_{year}": col in tables[year]
                for year in tables
            }
        }
        for col in all_columns
    ]

    display(spark.createDataFrame(comparison))


In [0]:

from pyspark.sql import functions as F
from functools import reduce

YEARS = [2018, 2019, 2020]

resultados = []

for year in YEARS:

    tabla = f"workspace.codi_bronze.deis_egresos_{year}"
    df = spark.table(tabla)

    # Año de egreso tal como aparece publicado
    ano = F.trim(F.col("ANO_EGRESO").cast("string"))

    resumen = df.agg(
        F.count("*").alias("total"),

        F.sum(
            F.when(ano == str(year), 1).otherwise(0)
        ).alias("anio_coincidente"),

        F.sum(
            F.when(ano == "*", 1).otherwise(0)
        ).alias("anio_asterisco"),

        F.sum(
            F.when(
                ano.isNull() | (ano == ""),
                1
            ).otherwise(0)
        ).alias("anio_nulo_vacio"),

        F.sum(
            F.when(
                ano.isNotNull()
                & (ano != "")
                & (ano != "*")
                & (ano != str(year)),
                1
            ).otherwise(0)
        ).alias("anio_otro")
    ).first()

    resultados.append({
        "year": year,
        "total": resumen["total"],
        "anio_coincidente": resumen["anio_coincidente"],
        "anio_asterisco": resumen["anio_asterisco"],
        "anio_nulo_vacio": resumen["anio_nulo_vacio"],
        "anio_otro": resumen["anio_otro"]
    })

df_auditoria = spark.createDataFrame(resultados)

df_auditoria = df_auditoria.withColumn(
    "pct_asterisco",
    F.round(
        F.col("anio_asterisco")
        / F.col("total") * 100,
        2
    )
)

display(df_auditoria.orderBy("year"))
